In [1]:
import pandas as pd
import numpy as np

In [2]:
men_df = pd.read_csv("../data/m_tournament_training_dataset.csv")


In [3]:
print(men_df.shape)
print(men_df.columns.tolist())

(2898, 20)
['Season', 'Team1ID', 'Team2ID', 'Target', 'WinPctDiff', 'SeedNumDiff', 'NetRatingDiff', 'OffEffDiff', 'DefEffDiff', 'MarginDiff', 'ReboundPctDiff', 'TurnoverPctDiff', 'FGPctDiff', 'ThreePctDiff', 'FTPctDiff', 'RankingDiff', 'PossessionsDiff', 'TurnoverMarginDiff', 'ReboundMarginDiff', 'AssistTurnoverRatioDiff']


Helper function to avoid divide-by-zero issues

In [4]:
def safe_divide(a, b):
    return np.where(np.abs(b) < 1e-8, 0, a / b)

### Let's create now advanced engineered features

#### Strength Interaction features

In [5]:
men_df["Seed_Rank_Interaction"] = men_df["SeedNumDiff"] * men_df["RankingDiff"]
men_df["WinPct_NetRating_Interaction"] = men_df["WinPctDiff"] * men_df["NetRatingDiff"]
men_df["Margin_Ranking_Interaction"] = men_df["MarginDiff"] * men_df["RankingDiff"]


#### Efficiency based interactions

In [6]:
men_df["OffDefGap"] = men_df["OffEffDiff"] - men_df["DefEffDiff"]
men_df["TotalEfficiencyGap"] = men_df["OffEffDiff"] + men_df["DefEffDiff"]
men_df["NetRating_Margin_Interaction"] = men_df["NetRatingDiff"] * men_df["MarginDiff"]
men_df["OffEff_DefEff_Product"] = men_df["OffEffDiff"] * men_df["DefEffDiff"]


#### Shooting profile interaction

In [7]:
men_df["ShootingEfficiencyScore"] = (
        men_df["FGPctDiff"] +
        men_df["ThreePctDiff"] +
        men_df["FTPctDiff"]
)

men_df["WeightedShootingScore"] = (
        0.5 * men_df["FGPctDiff"] +
        0.3 * men_df["ThreePctDiff"] +
        0.2 * men_df["FTPctDiff"]
)

men_df["FG_Three_Interaction"] = men_df["FGPctDiff"] * men_df["ThreePctDiff"]
men_df["FG_FT_Interaction"] = men_df["FGPctDiff"] * men_df["FTPctDiff"]
men_df["Three_FT_Interaction"] = men_df["ThreePctDiff"] * men_df["FTPctDiff"]


#### Ball control

In [8]:
men_df["ControlScore"] = (
        men_df["TurnoverPctDiff"] +
        men_df["TurnoverMarginDiff"] +
        men_df["AssistTurnoverRatioDiff"]
)

men_df["ReboundControlScore"] = (
        men_df["ReboundPctDiff"] +
        men_df["ReboundMarginDiff"]
)

men_df["PossessionControlInteraction"] = (
        men_df["PossessionsDiff"] * men_df["TurnoverPctDiff"]
)

men_df["ReboundTurnoverCombo"] = (
        men_df["ReboundPctDiff"] - men_df["TurnoverPctDiff"]
)

#### Dominance

In [9]:
men_df["DominanceScore"] = men_df["MarginDiff"] * men_df["WinPctDiff"]
men_df["SeedAdjustedDominance"] = safe_divide(men_df["MarginDiff"], men_df["SeedNumDiff"] + 1)
men_df["RankingAdjustedNetRating"] = safe_divide(men_df["NetRatingDiff"], men_df["RankingDiff"] + 1)


Nonlinear features

In [10]:
men_df["SeedNumDiff_Squared"] = men_df["SeedNumDiff"] ** 2
men_df["NetRatingDiff_Squared"] = men_df["NetRatingDiff"] ** 2
men_df["MarginDiff_Squared"] = men_df["MarginDiff"] ** 2
men_df["RankingDiff_Squared"] = men_df["RankingDiff"] ** 2

men_df["AbsSeedNumDiff"] = men_df["SeedNumDiff"].abs()
men_df["AbsNetRatingDiff"] = men_df["NetRatingDiff"].abs()
men_df["AbsMarginDiff"] = men_df["MarginDiff"].abs()
men_df["AbsRankingDiff"] = men_df["RankingDiff"].abs()

#### Threshold / indicator features
#### XGBoost can benefit from these simple flags

In [11]:
men_df["SeedAdvantageFlag"] = (men_df["SeedNumDiff"] < 0).astype(int)
men_df["NetRatingAdvantageFlag"] = (men_df["NetRatingDiff"] > 0).astype(int)
men_df["OffEffAdvantageFlag"] = (men_df["OffEffDiff"] > 0).astype(int)
men_df["DefEffAdvantageFlag"] = (men_df["DefEffDiff"] < 0).astype(int)  # lower defensive efficiency allowed is better depending on your definition
men_df["MarginAdvantageFlag"] = (men_df["MarginDiff"] > 0).astype(int)
men_df["RankingAdvantageFlag"] = (men_df["RankingDiff"] < 0).astype(int)  # assuming lower ranking is better


#### Combined advantage count

In [12]:
men_df["AdvantageCount"] = (
        men_df["SeedAdvantageFlag"] +
        men_df["NetRatingAdvantageFlag"] +
        men_df["OffEffAdvantageFlag"] +
        men_df["DefEffAdvantageFlag"] +
        men_df["MarginAdvantageFlag"] +
        men_df["RankingAdvantageFlag"]
)

In [13]:
men_df.replace([np.inf, -np.inf], 0, inplace=True)

,Season,Team1ID,Team2ID,Target,WinPctDiff,SeedNumDiff,NetRatingDiff,OffEffDiff,DefEffDiff,MarginDiff,...,AbsNetRatingDiff,AbsMarginDiff,AbsRankingDiff,SeedAdvantageFlag,NetRatingAdvantageFlag,OffEffAdvantageFlag,DefEffAdvantageFlag,MarginAdvantageFlag,RankingAdvantageFlag,AdvantageCount
0,2003,1421,1411,1,-0.151724,0.0,-0.118699,-0.021731,0.096968,-9.208046,...,0.118699,9.208046,16.0,0,0,0,0,0,0,0
1,2003,1112,1436,1,0.237685,-15.0,0.120916,0.081979,-0.038936,10.309113,...,0.120916,10.309113,145.0,1,1,1,1,1,1,6
2,2003,1113,1272,1,-0.172414,3.0,-0.023829,0.040148,0.063977,-1.896552,...,0.023829,1.896552,22.0,0,0,1,0,0,0,1
3,2003,1141,1166,1,-0.085684,5.0,-0.126746,-0.040490,0.086256,-8.805643,...,0.126746,8.805643,17.0,0,0,0,0,0,0,0
4,2003,1143,1301,1,0.124138,-1.0,0.002024,-0.020811,-0.022835,0.324138,...,0.002024,0.324138,18.0,1,1,0,1,1,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2893,2025,1277,1120,0,-0.030303,1.0,-0.042985,-0.082566,-0.039581,-3.333333,...,0.042985,3.333333,4.0,0,0,0,1,0,0,1
2894,2025,1397,1222,0,-0.088235,1.0,-0.071351,-0.015717,0.055633,-4.029412,...,0.071351,4.029412,5.0,0,0,0,0,0,0,0
2895,2025,1120,1196,0,-0.033868,0.0,-0.026826,0.004642,0.031467,-1.934046,...,0.026826,1.934046,1.0,0,0,1,0,0,0,1
2896,2025,1181,1222,0,0.029412,0.0,0.060774,0.079816,0.019043,5.058824,...,0.060774,5.058824,1.0,0,1,1,0,1,0,3


In [14]:
print("Total missing values:", men_df.isna().sum().sum())

Total missing values: 0


In [16]:
new_features = [
    "Seed_Rank_Interaction",
    "WinPct_NetRating_Interaction",
    "Margin_Ranking_Interaction",
    "OffDefGap",
    "TotalEfficiencyGap",
    "NetRating_Margin_Interaction",
    "OffEff_DefEff_Product",
    "ShootingEfficiencyScore",
    "WeightedShootingScore",
    "FG_Three_Interaction",
    "FG_FT_Interaction",
    "Three_FT_Interaction",
    "ControlScore",
    "ReboundControlScore",
    "PossessionControlInteraction",
    "ReboundTurnoverCombo",
    "DominanceScore",
    "SeedAdjustedDominance",
    "RankingAdjustedNetRating",
    "SeedNumDiff_Squared",
    "NetRatingDiff_Squared",
    "MarginDiff_Squared",
    "RankingDiff_Squared",
    "AbsSeedNumDiff",
    "AbsNetRatingDiff",
    "AbsMarginDiff",
    "AbsRankingDiff",
    "SeedAdvantageFlag",
    "NetRatingAdvantageFlag",
    "OffEffAdvantageFlag",
    "DefEffAdvantageFlag",
    "MarginAdvantageFlag",
    "RankingAdvantageFlag",
    "AdvantageCount"
]

In [17]:
print(men_df[new_features].head())

   Seed_Rank_Interaction  WinPct_NetRating_Interaction  \
0                    0.0                      0.018010   
1                 2175.0                      0.028740   
2                   66.0                      0.004108   
3                   85.0                      0.010860   
4                   18.0                      0.000251   

   Margin_Ranking_Interaction  OffDefGap  TotalEfficiencyGap  \
0                 -147.328736  -0.118699            0.075237   
1                -1494.821429   0.120916            0.043043   
2                  -41.724138  -0.023829            0.104125   
3                 -149.695925  -0.126746            0.045765   
4                   -5.834483   0.002024           -0.043646   

   NetRating_Margin_Interaction  OffEff_DefEff_Product  \
0                      1.092988              -0.002107   
1                      1.246532              -0.003192   
2                      0.045193               0.002569   
3                      1.116078   

In [18]:
output_path = "../data/m_tournament_training_dataset_advanced.csv"
men_df.to_csv(output_path, index=False)